# 06 — CLI Chatbot with MCP

This notebook walks through building a CLI chatbot that uses an MCP server, MCP client, and the Anthropic API.

## Learning Objectives

- Understand the agentic loop pattern
- Connect a client to a server
- Send tool schemas to Claude
- Execute tool calls and return results
- Build a complete conversation flow

## Architecture

```
User (CLI input)
  │
  ▼
Conversation Runner
  ├─► MCP Client ──► MCP Server (tools)
  └─► Anthropic API (Claude)
  │
  ▼
User (CLI output)
```

The **Conversation Runner** implements an agentic loop:
1. Send user message + tool schemas to Claude
2. If Claude returns `tool_use` → execute via MCP client → loop
3. If Claude returns text → show to user → wait for next input

In [ ]:
# The agentic loop — core pattern of the CLI chatbot

AGENTIC_LOOP = '''
async def run_conversation(user_message: str, messages: list, tools: list, client):
    """Run one turn of the agentic loop."""
    messages.append({"role": "user", "content": user_message})

    while True:
        # Send to Claude with available tools
        response = anthropic_client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=8000,
            messages=messages,
            tools=tools,
        )

        # Add Claude's response to conversation history
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "tool_use":
            # Claude wants to use a tool — execute it
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = await client.call_tool(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(
                            [item.text for item in result.content]
                        ),
                    })

            # Send tool results back to Claude
            messages.append({"role": "user", "content": tool_results})
        else:
            # Claude returned text — extract and return
            text = "\\n".join(
                block.text for block in response.content if block.type == "text"
            )
            return text
'''

print("The agentic loop:")
print("  1. Append user message to history")
print("  2. Send history + tools to Claude")
print("  3. If stop_reason == 'tool_use':")
print("     a. Execute each tool via MCP client")
     b. Append tool results as 'user' message")
print("     c. LOOP back to step 2")
print("  4. If stop_reason == 'end_turn':")
print("     a. Extract text from response")
print("     b. Return to user")

## Tool Schema Conversion

MCP tools need to be converted to Anthropic API format:

In [ ]:
# MCP tool → Anthropic API tool format

# What MCP returns from list_tools():
mcp_tool = {
    "name": "read_doc_contents",
    "description": "Read the contents of a document",
    "inputSchema": {  # Note: camelCase from MCP
        "type": "object",
        "properties": {
            "doc_id": {"type": "string", "description": "Document ID"}
        },
        "required": ["doc_id"],
    },
}

# What Anthropic API expects:
anthropic_tool = {
    "name": mcp_tool["name"],
    "description": mcp_tool["description"],
    "input_schema": mcp_tool["inputSchema"],  # Note: snake_case for Anthropic
}

import json
print("Anthropic API tool format:")
print(json.dumps(anthropic_tool, indent=2))

## main.py Structure

In [ ]:
MAIN_CODE = '''
import asyncio
import sys
import os
from dotenv import load_dotenv
from contextlib import AsyncExitStack

load_dotenv()

async def main():
    # 1. Create MCP client and connect to server
    async with AsyncExitStack() as stack:
        client = await stack.enter_async_context(
            MCPClient(command="python", args=["server.py"])
        )

        # 2. Get tool schemas from MCP server
        tools = await get_tool_schemas(client)

        # 3. Run conversation loop
        messages = []
        while True:
            user_input = input("You: ")
            if user_input.lower() in ("quit", "exit"):
                print("Goodbye!")
                break

            response = await run_conversation(
                user_input, messages, tools, client
            )
            print(f"Claude: {response}")

if __name__ == "__main__":
    asyncio.run(main())
'''

print("main.py flow:")
print("  1. Load environment variables")
print("  2. Create MCP client → connect to server")
print("  3. Fetch tool schemas from server")
print("  4. Enter input loop")
print("  5. For each input: run agentic loop")
print("  6. Print Claude's response")
print("  7. Repeat until 'quit'")

## Running the Chatbot

```bash
# From the repo root:
cd projects/cli_chatbot
python main.py

# Or from repo root:
python projects/cli_chatbot/main.py
```

### Example Conversation

```
You: What documents do you have access to?
Claude: Let me check the available documents.
       [Calls read_doc_contents for each, or a list resource]
       I have access to:
       - report.pdf: Details about a condenser tower
       - plan.md: Implementation steps
       - spec.txt: Technical requirements

You: Read the report for me
Claude: [Calls read_doc_contents("report.pdf")]
       The report details the state of a 20m condenser tower.

You: quit
Goodbye!
```

## Exercise

1. Trace through what happens when the user types: "Edit the report to change '20m' to '20-meter'"
2. How many Claude API calls are made in that conversation turn?
3. What messages are in the conversation history after that turn?

In [ ]:
# Exercise: Trace the conversation

print("User: Edit the report to change '20m' to '20-meter'")
print()
print("Step 1: Append user message to history")
print("  messages = [{role: 'user', content: 'Edit the report...'}]")
print()
print("Step 2: Send to Claude API (call #1)")
print("  Claude response: stop_reason='tool_use'")
print("  Tool: edit_document(doc_id='report.pdf', old_str='20m', new_str='20-meter')")
print()
print("Step 3: Execute tool via MCP client")
print("  Client → Server: call_tool('edit_document', {...})")
print("  Server: performs replacement, returns success")
print()
print("Step 4: Append tool result to messages")
print("  messages += [{role: 'user', content: [tool_result]}]")
print()
print("Step 5: Send to Claude API (call #2)")
print("  Claude response: stop_reason='end_turn'")
print("  Text: 'I\'ve updated the report...'")
print()
print("Total Claude API calls: 2")
print("Final messages in history: 4")
print("  1. user: original query")
print("  2. assistant: tool_use block")
print("  3. user: tool_result")
print("  4. assistant: final text response")